# Démo — Système d'Allocation Dynamique

Simule l'arrivée de patients en temps réel avec :
- File de priorité **ESI FIFO**
- **Zone de quarantaine** pour patients non allouables
- **Réinjection automatique** à chaque libération de ressources


## 1. Chargement du Modèle Entraîné

Avant de lancer la simulation, on charge tous les artefacts produits par `dnn_patient_allocation.ipynb` :

| Artefact | Rôle |
| --- | --- |
| `dnn_weights.pt` | Poids du DNN PyTorch entraîné |
| `scaler_p.pkl` | Normaliseur des features patients |
| `scaler_h.pkl` | Normaliseur des features hôpitaux + distance |
| `label_encoder.pkl` | Encodeur des noms d'hôpitaux |

> `allocation_system.py` est importé pour accéder à `AllocationEngine`, `Patient`, `QuarantineZone`, etc.

In [ ]:
import sys, pickle, torch, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from allocation_system import (
    AllocationEngine, Patient, PatientQueue, QuarantineZone,
    HospitalState, PATIENT_FEATURES, HOSP_RES_COLS
)

# Charger le modèle entraîné
OUT = ROOT / 'output'
DEVICE = torch.device('cpu')

import torch.nn as nn
class PatientAllocationDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64),       nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear( 64,  32),       nn.ReLU(),
            nn.Linear( 32,   1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

INPUT_DIM = len(PATIENT_FEATURES) + len(HOSP_RES_COLS) + 1
model = PatientAllocationDNN(INPUT_DIM)
model.load_state_dict(torch.load(OUT/'dnn_weights.pt', map_location=DEVICE))
model.eval()

with open(OUT/'scaler_p.pkl','rb') as f: scaler_p = pickle.load(f)
with open(OUT/'scaler_h.pkl','rb') as f: scaler_h = pickle.load(f)
with open(OUT/'label_encoder.pkl','rb') as f: le = pickle.load(f)

patients  = pd.read_excel(ROOT.parent/'xgboost_model'/'patients_1000_ULTRA_COMPLET.xlsx')
hospitals = pd.read_excel(ROOT/'Book1.xlsx')
print('Système chargé.')

## 2. Initialisation du Moteur d'Allocation

`AllocationEngine` est le chef d'orchestre du système. À l'initialisation :

- Crée un objet `HospitalState` pour **chacun des 22 hôpitaux** avec ses ressources initiales
- Instancie une **file principale** (`PatientQueue` triée par ESI)
- Instancie une **zone de quarantaine** (`QuarantineZone`) vide

Le dashboard initial montre tous les hôpitaux à **0% de saturation** (ressources pleines).

In [ ]:
engine = AllocationEngine(
    model=model,
    scaler_p=scaler_p,
    scaler_h=scaler_h,
    label_encoder=le,
    hospitals_df=hospitals,
    device=DEVICE,
)
print('Moteur d\'allocation initialisé.')
print(engine.dashboard().to_string(index=False))

## 3. Simulation — Arrivée de 50 Patients

On sélectionne 50 patients aléatoires et on les ajoute à la **file principale**.

**Règle de priorité ESI (Emergency Severity Index) :**

| ESI | Urgence | Traitement |
| --- | --- | --- |
| 1 | Critique (réanimation immédiate) | Traité en **premier** |
| 2 | Urgence majeure | Traité en **deuxième** |
| 3 | Urgence modérée | Traité en **troisième** |
| 4 | Urgence mineure | Traité en **quatrième** |
| 5 | Non urgent | Traité en **dernier** |

> À ESI égal, l'ordre FIFO (heure d'arrivée) est respecté grâce au `arrival_counter`.

In [ ]:
# Admettre 50 patients dans la file principale (ESI FIFO)
sample = patients.sample(50, random_state=42).reset_index(drop=True)
for _, row in sample.iterrows():
    p = Patient(id=int(row['ID']), features=row.to_dict())
    engine.admit(p)

print(f'File principale : {len(engine.main_queue)} patients')
print(f'ESI du prochain patient : {engine.main_queue.peek_esi()}')

## 4. Traitement de la File — Allocation par le DNN

**`engine.process_all()`** vide la file principale en traitant chaque patient dans l'ordre ESI :

```
Pour chaque patient (par ordre ESI) :
  1. Calculer le score DNN pour chacun des 22 hôpitaux
  2. Filtrer les hôpitaux saturés (≥ 95% lits occupés)
  3. Vérifier la faisabilité des ressources en temps réel
  4a. Si hôpital trouvé → allouer + consommer les ressources
  4b. Sinon → placer le patient en zone de quarantaine
```

In [ ]:
print('Traitement de la file...')
log = engine.process_all()
stats = engine.global_stats()
print('\n=== STATISTIQUES ===')
for k, v in stats.items(): print(f'  {k}: {v}')
print('\nDashboard hôpitaux:')
print(engine.dashboard().to_string(index=False))

## 5. Visualisation des Résultats

Deux graphiques sont générés :

**Graphique gauche — Répartition des allocations par hôpital**
Montre combien de patients ont été envoyés vers chaque hôpital.

**Graphique droite — Taux de saturation**
- 🟢 Vert : < 50% (hôpital disponible)
- 🟠 Orange : 50–80% (capacité partielle)
- 🔴 Rouge : > 80% (proche de la saturation)
- Ligne rouge pointillée : seuil de 95% (= saturation critique → hôpital exclu)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Répartition des allocations par hôpital
df_log = engine.allocation_summary()
if not df_log.empty:
    counts = df_log['hospital'].value_counts()
    counts.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Patients alloués par hôpital')
    axes[0].set_xlabel('Nombre de patients')

# Saturation des hôpitaux
dash = engine.dashboard()
colors = ['tomato' if s > 80 else 'orange' if s > 50 else 'mediumseagreen'
          for s in dash['saturation_%']]
axes[1].barh(dash['hospital'], dash['saturation_%'], color=colors)
axes[1].axvline(95, color='red', linestyle='--', label='Seuil saturation')
axes[1].set_title('Taux de saturation des hôpitaux (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig(OUT/'simulation_results.png', dpi=120)
plt.show()

## 6. Libération de Ressources & Réinjection Dynamique

Simule la **sortie de patients** d'un hôpital (fin de traitement) :

**`engine.release_patient(patient, hopital)`** effectue en séquence :
1. **Libère** les ressources consommées par ce patient dans l'hôpital (`HospitalState.release()`)
2. Déclenche automatiquement **`reinject()`** sur la zone de quarantaine
3. Tente de réallouer chaque patient en quarantaine (dans l'ordre ESI) avec les nouvelles ressources
4. Les patients encore non allouables **retournent en quarantaine**

> Ce mécanisme simule le flux temps-réel du schéma d'architecture : la **réinjection** depuis la buffer zone.

In [ ]:
print(f'Patients en quarantaine AVANT libération: {len(engine.quarantine)}')

# Simuler la sortie de 5 patients de l'hôpital le plus chargé
dash = engine.dashboard().sort_values('patients_allocated', ascending=False)
busiest_hospital = dash.iloc[0]['hospital']
print(f'Hôpital le plus chargé: {busiest_hospital}')

df_log = engine.allocation_summary()
if not df_log.empty:
    discharged = df_log[df_log['hospital'] == busiest_hospital].head(5)
    for _, alloc in discharged.iterrows():
        # Reconstruire le patient depuis ses données
        pid  = alloc['patient_id']
        prow = patients[patients['ID'] == pid]
        if prow.empty: continue
        p = Patient(id=pid, features=prow.iloc[0].to_dict())
        result = engine.release_patient(p, busiest_hospital)
        print(f'  Patient {pid} sorti → réinjection: {result}')

print(f'\nPatients en quarantaine APRÈS réinjection: {len(engine.quarantine)}')
print('\nStats finales:', engine.global_stats())

## 7. Analyse de la Zone de Quarantaine

La `QuarantineZone` maintient un **log horodaté** de toutes les admissions avec :
- `patient_id` : identifiant du patient
- `ESI` : niveau d'urgence (pour voir si les patients critiques sont bien réinjectés en priorité)
- `reason` : cause de mise en quarantaine (`no_feasible_hospital` ou `reinjection_failed`)
- `admitted_at` : timestamp d'entrée

**Idéalement**, les patients ESI 1 et 2 ne doivent jamais rester longtemps en quarantaine.

> Si beaucoup de patients ESI 1–2 sont en quarantaine, cela signale une **saturation critique** du système hospitalier.

In [ ]:
q_summary = engine.quarantine.summary()
if not q_summary.empty:
    print('Zone de quarantaine — patients en attente:')
    print(q_summary.to_string(index=False))
    fig, ax = plt.subplots(figsize=(7,4))
    q_summary['ESI'].value_counts().sort_index().plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Distribution ESI — Zone de quarantaine')
    ax.set_xlabel('Niveau ESI'); ax.set_ylabel('Nombre de patients')
    plt.tight_layout()
    plt.savefig(OUT/'quarantine_esi.png', dpi=120)
    plt.show()
else:
    print('Aucun patient en quarantaine — tous ont été alloués !')